In [1]:
import pandas as pd
import numpy as np

In [2]:
train_file = open('./LightGCN/train.txt', 'r')
test_file = open('./LightGCN/test.txt', 'r')

uid_ls, his_ls = [], []

for train_line, test_line in zip(train_file.readlines(), test_file.readlines()):
    tmp_train_line = train_line.strip().split(' ')
    tmp_test_line = test_line.strip().split(' ')

#     uid = int(tmp_train_line[0])
    uid = int(tmp_train_line[0]) + 1 #因为是从lightGCN从0开始编码，但是我们要把0当作padding

    assert tmp_train_line[0] == tmp_test_line[0]

    his = [int(iid)+1 for iid in tmp_train_line[1:]] + \
            [int(iid)+1 for iid in tmp_test_line[1:]]
    
    uid_ls.append(uid)
    his_ls.append(his)




In [3]:
use_all_his = pd.DataFrame(
    data = zip(uid_ls, his_ls), columns=['user_id', 'user_history']
)

In [4]:
use_all_his

,user_id,user_history
0,1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,2,"[19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3..."
2,3,"[554, 555, 556, 557, 558, 559, 560, 561, 562, ..."
3,4,"[561, 559, 600, 555, 566, 622, 573, 578, 588, ..."
4,5,"[625, 626, 627, 628, 629, 630, 631, 632, 633, ..."
...,...,...
31663,31664,"[32941, 36662, 28423, 11153, 37681, 3207, 3536..."
31664,31665,"[11125, 7366, 16965, 21421, 16885, 24961, 1789..."
31665,31666,"[8846, 9847, 10252, 26225, 14973, 11528, 13748..."
31666,31667,"[37463, 29817, 2356, 23560, 9633, 13381, 6401,..."


In [23]:
import ast
df = use_all_his.copy()

# Flatten all lists in 'user_history' column
all_ids = [item for sublist in df['user_history'] for item in sublist]

# Get unique values and count
unique_ids = set(all_ids)
unique_ids_count = len(unique_ids)

print("Total number of unique IDs:", unique_ids_count)


Total number of unique IDs: 38048


In [25]:
num_users = df['user_id'].nunique()
print("Number of unique users:", num_users)

Number of unique users: 31668


In [6]:
use_all_his.to_csv('./dataset/user_with_full_his.tsv', sep='\t', index=False)

In [8]:
import numpy as np

all_user_item_inter_pd = use_all_his.copy()

all_users = all_user_item_inter_pd.shape[0]
test_val_users = np.random.choice(all_users, int(all_users * 0.2), replace=False)
test = test_val_users[:len(test_val_users)//2]
val = test_val_users[len(test_val_users)//2:]

training_data = all_user_item_inter_pd[
    ~all_user_item_inter_pd['user_id'].isin(test_val_users)
].copy()
val_data = all_user_item_inter_pd[
    all_user_item_inter_pd['user_id'].isin(val)
].copy()
test_data = all_user_item_inter_pd[
    all_user_item_inter_pd['user_id'].isin(test)
].copy()

training_data.shape, val_data.shape, test_data.shape

((25335, 2), (3167, 2), (3166, 2))

In [9]:
val_data.head()

,user_id,user_history
18,19,"[761, 766, 88, 1000, 765, 828, 763, 1001, 1002..."
21,22,"[1179, 1180, 1181, 1056, 1073, 1182, 1183, 118..."
22,23,"[1135, 1239, 1110, 1240, 1241, 1242, 1243, 124..."
33,34,"[1529, 817, 663, 784, 1683, 650, 88, 689, 1530..."
45,46,"[1, 650, 2018, 2750, 770, 2815, 658, 1861, 162..."


In [10]:
val_data['given_user_history'] = val_data['user_history'].apply(
    lambda x:
        x[:int(0.8*len(x))] 
        # if len(x) >= 10
        # else x[:-2]
)
val_data['predicting_items'] = val_data['user_history'].apply(
    lambda x:
        x[int(0.8*len(x)):] 
        # if len(x) >= 10
        # else x[-2:]
)

In [11]:
test_data['given_user_history'] = test_data['user_history'].apply(
    lambda x:
        x[:int(0.8*len(x))] 
        # if len(x) >= 10
        # else x[:-2]
)
test_data['predicting_items'] = test_data['user_history'].apply(
    lambda x:
        x[int(0.8*len(x)):] 
        # if len(x) >= 10
        # else x[-2:]
)

In [12]:
val_data.head()

,user_id,user_history,given_user_history,predicting_items
18,19,"[761, 766, 88, 1000, 765, 828, 763, 1001, 1002...","[761, 766, 88, 1000, 765, 828, 763, 1001, 1002...","[2536, 1456, 1514, 2366, 21402, 37421, 7605, 4..."
21,22,"[1179, 1180, 1181, 1056, 1073, 1182, 1183, 118...","[1179, 1180, 1181, 1056, 1073, 1182, 1183, 118...","[13780, 6693, 1382, 5534, 11278, 1647, 1034, 1..."
22,23,"[1135, 1239, 1110, 1240, 1241, 1242, 1243, 124...","[1135, 1239, 1110, 1240, 1241, 1242, 1243, 124...","[1158, 5240, 15336, 7828, 1660, 6576, 1479, 1054]"
33,34,"[1529, 817, 663, 784, 1683, 650, 88, 689, 1530...","[1529, 817, 663, 784, 1683, 650, 88, 689, 1530...","[2741, 772, 773, 2867]"
45,46,"[1, 650, 2018, 2750, 770, 2815, 658, 1861, 162...","[1, 650, 2018, 2750, 770, 2815, 658, 1861, 162...","[9789, 7039, 3279, 1035, 1104, 1044, 1037, 280..."


In [13]:
val_data.drop(columns=['user_history']).to_csv(
    './dataset/validation.tsv', sep='\t', index=False
)
test_data.drop(columns=['user_history']).to_csv(
    './dataset/test.tsv', sep='\t', index=False
)

In [14]:
training_data.to_csv(
    './dataset/training_with_full_his.tsv', sep='\t', index=False
)

In [15]:
training_data['user_history'].loc[0]

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 796,
 695,
 1532,
 14518,
 8785]

In [16]:
import pandas as pd

training_his = training_data
# training_his = pd.read_csv(
#     './dataset/training.tsv', sep='\t'
# )

uid_ls, user_his_ls, next_item_ls = [], [], []

for _, user_table in training_his.iterrows():
    cur_reco_his = user_table['user_history']
    # cur_reco_his = eval(user_table['user_history'])
    for i in range(1, len(cur_reco_his)):
        user_his_ls.append(cur_reco_his[:i])
        uid_ls.append(int(user_table['user_id']))
        next_item_ls.append(cur_reco_his[i])

training_pd = pd.DataFrame(
    data=zip(uid_ls, user_his_ls, next_item_ls), columns=['user_id', 'user_history', 'next_item']
)

In [17]:
training_pd.head(), training_pd.shape

(   user_id     user_history  next_item
 0        1              [1]          2
 1        1           [1, 2]          3
 2        1        [1, 2, 3]          4
 3        1     [1, 2, 3, 4]          5
 4        1  [1, 2, 3, 4, 5]          6,
 (1230559, 3))

In [18]:
training_pd.to_csv('./dataset/training.tsv', sep='\t', index=False)